# Artificial Intelligence Assignment #2-2<br> Training Vision Transformers (PyTorch)

Copyright (C) Computer Science & Engineering, Soongsil University. This material is for educational uses only. Some contents are based on the material provided by other paper/book authors and may be copyrighted by them (September 2025.)

**For understanding of this work, please carefully look at given PDF file.**

Now, you're going to leave behind your implementations and instead migrate to one of popular deep learning frameworks, **PyTorch**. <br>
In this notebook, you will learn to understand and build the basic components of Vision Tranformer(ViT). Then, you will try to classify images in the FashionMNIST datatset and explore the effects of different components of ViTs.
<br>
There are **2 sections**, and in each section, you need to follow the instructions to complete the skeleton codes and explain them.

**Note**: certain details are missing or ambiguous on purpose, in order to test your knowledge on the related materials. However, if you really feel that something essential is missing and cannot proceed to the next step, then contact the teaching staff with clear description of your problem.

### Submitting your work:
<font color=red>**DO NOT clear the final outputs**</font> so that TAs can grade both your code and results.

### Some helpful tutorials and references for assignment #2-2:
- [1] Pytorch official documentation [[link]](https://pytorch.org/docs/stable/index.html).
- [2] Stanford CS231n lectures [[link]](http://cs231n.stanford.edu/).
- [3] Alexey Dosovitskiy et al., "An Image is Worth 16 x 16 Words: Transformers for Image Recognition at Scale", ICLR 2021 [[pdf]](https://arxiv.org/pdf/2010.11929.pdf).

## 1. Building Vision Transformer
Here, you will build the basic components of Vision Transformer(ViT). <br>

![Vision Transformer](imgs/ViT.png)

Using the explanation and code provided as guidance, <br>
Define each component of ViT. <br>


#### ViT architecture:
* ViT model consists with input patch embedding, positional embeddings, transformer encoder, etc.
* Patch embedding
* Positional embeddings
* Transformer encoder with
    * Attention module
    * MLP module

In [1]:
import torch
import torch.nn as nn

##### Patch Embed

**Initialization**: When you create an instance of the PatchEmbedding class, you specify the image_size, patch_size, and in_channels. image_size is the height and width of the input image, patch_size is the size of each patch, and in_channels is the number of input image channels (e.g., 3 for RGB images).

**Convolutional Projection**: Inside the PatchEmbedding class, a 2D convolutional layer (nn.Conv2d) is used to perform a patch-based projection. This convolutional layer has a kernel size of patch_size, which defines the size of each patch, and a stride of patch_size, which ensures that patches do not overlap. The convolutional layer effectively extracts image patches.

**Reshaping**: After the convolutional projection, the output tensor is reshaped using view. It is transformed from a 4D tensor with dimensions (batch_size, in_channels, H, W) to a 3D tensor with dimensions (batch_size, num_patches, patch_dim). num_patches is the total number of non-overlapping patches in the image, and patch_dim is the number of output channels from the convolutional layer.

In [2]:
class PatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """

    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        num_patches = (img_size // patch_size) * (img_size // patch_size)
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = num_patches

        ##############################################################################
        #                           IMPLEMENT YOUR CODE                              #
        ##############################################################################

        #nnConv2d(input channels, output channels, kernel size, stride)
        self.proj = nn.Conv2d(in_channels=in_chans, out_channels=embed_dim, kernel_size=patch_size, stride=patch_size)


        ##############################################################################
        #                              END YOUR CODE                                 #
        ##############################################################################
    def forward(self, x):
        ##############################################################################
        #                           IMPLEMENT YOUR CODE                              #
        ##############################################################################
        x = self.proj(x) # Conv2d 적용
        x = x.flatten(2) # 2D그리드를 1D sequence로 펼침(196개의 patch가 일렬로)
        x = x.transpose(1,2) # transformer 입력 표준인 (Batch, sequecne, featuer)로 순서를 바꿈

        ##############################################################################
        #                              END YOUR CODE                                 #
        ##############################################################################
        return x # output dimension must be: (batch size, number of patches, embed_dim)

##### Attention

**Initialization**
* dim: The input dimension of the sequence. This is the dimensionality of the queries, keys, and values.
* num_heads: The number of attention heads to use. Multi-head attention allows the model to focus on different parts of the input simultaneously.

**Linear Projections (qkv and proj)**: The qkv linear layer takes the input sequence and projects it into three parts: queries (q), keys (k), and values (v). The output of this layer has a shape of (batch_size, sequence_length, 3 * dim).

**Forward Pass (forward method)**: In the forward pass, the input tensor x is processed through the attention mechanism. Here's what happens:<br>
* The linear projection qkv is applied to x, producing a tensor of shape (batch_size, sequence_length, 3 * dim).|
* This tensor is reshaped to have dimensions (batch_size, sequence_length, 3, num_heads, head_dim). The permute operation rearranges the dimensions to (3, batch_size, num_heads, sequence_length, head_dim), making it suitable for multi-head attention.
* The three parts, q, k, and v, are extracted from the reshaped tensor.
* The attention scores are computed by taking the dot product of queries q and keys k. The result is scaled by self.scale.
* The attention scores are passed through a softmax activation along the last dimension (sequence_length), producing attention weights.
* The weighted sum of values v is computed using the attention weights.
* The result is transposed and reshaped to its original shape, and then passed through the proj linear layer.
* The final output is returned.

In [3]:
class Attention(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        ##############################################################################
        #                           IMPLEMENT YOUR CODE                              #
        ##############################################################################

        #입력 x를 Q,K,V로 한번에 변환하고 multi-head처리를 위해 reshape를 해서 차원을 나눈 뒤 차원의 순서를 바꿈
        # batch와 heads를 붙여놔야 병렬로 행렬 곱셈이 가능하기 때문
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)

        #q,k,v를 분리
        q, k, v = qkv[0], qkv[1], qkv[2]

        #Attention score 계산
        attn = (q @ k.transpose(-2, -1)) * self.scale

        #softmax 적용
        attn = attn.softmax(dim=-1)
        ##############################################################################
        #                              END YOUR CODE                                 #
        ##############################################################################

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)

        return x # output dimension must be: (batch size, number of patches, embed_dim)

##### MLP

The MLP module must consist of three layers:
* fully conncted layer 1
* activation layer
* fully conncted layer 2

In [4]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        ##############################################################################
        #                           IMPLEMENT YOUR CODE                              #
        ##############################################################################

        # FC layer1 - 확장
        self.fc1 = nn.Linear(in_features, hidden_features)

        # Activation layer 기본은 GELU로 됨
        self.act = act_layer()

        # FC layer2 - 복구/압
        self.fc2 = nn.Linear(hidden_features, out_features)
        ##############################################################################
        #                              END YOUR CODE                                 #
        ##############################################################################

    def forward(self, x):
        ##############################################################################
        #                           IMPLEMENT YOUR CODE                              #
        ##############################################################################

        #위에서 만든 FC layer, activation function으로 forward pass를 함
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        ##############################################################################
        #                              END YOUR CODE                                 #
        ##############################################################################
        return x # output dimension must be: (batch size, number of patches, out_features)

##### Transformer Block
The transformer block contains the attention module and MLP module which have residual connections.
Refer to the following image and build the forward pass.

![Transformer Block](imgs/TransformerBlock.png)

In [5]:
class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim,
                       act_layer=act_layer)

    def forward(self, x):
        ##############################################################################
        #                           IMPLEMENT YOUR CODE                              #
        ##############################################################################

        # input data를 먼저 normalization하고 self attention 수행. 그 후 skip connection을 함. 여기서 norm은 layer norm
        x = x + self.attn(self.norm1(x))

        # 바로 위에서 만든 data를 normalization하고 MLP를 통과시켜 각 patch의 feature를 가공하여 skip connection을 함
        x = x + self.mlp(self.norm2(x))

        ##############################################################################
        #                              END YOUR CODE                                 #
        ##############################################################################
        return x


##### Vision Transformer

Using all the components that you built above, **complete** the vision transformer class.

### torch.cat

Concatenates the given sequence of tensors along the specified dimension. All tensors must either have the same shape (except in the concatenating dimension) or be a 1-D empty tensor with size (0,).

`torch.cat()` can be seen as an inverse operation for `torch.split()` and `torch.chunk()`.

`torch.cat()` can be best understood via examples.

#### Parameters

- **tensors** (sequence of Tensors): any Python sequence of tensors of the same type. Non-empty tensors provided must have the same shape, except in the concatenating dimension.

- **dim** (int, optional): the dimension over which the tensors are concatenated.

#### Keyword Arguments

- **out** (Tensor, optional): the output tensor.

In [6]:
# example
x = torch.randn(2, 3)
x

tensor([[-1.4895,  1.3436, -0.2405],
        [ 0.0212, -0.5915, -0.5571]])

In [7]:
exam1 = torch.cat((x, x, x), 0)
exam1

tensor([[-1.4895,  1.3436, -0.2405],
        [ 0.0212, -0.5915, -0.5571],
        [-1.4895,  1.3436, -0.2405],
        [ 0.0212, -0.5915, -0.5571],
        [-1.4895,  1.3436, -0.2405],
        [ 0.0212, -0.5915, -0.5571]])

In [8]:
exam2 = torch.cat((x, x, x), 1)
exam2

tensor([[-1.4895,  1.3436, -0.2405, -1.4895,  1.3436, -0.2405, -1.4895,  1.3436,
         -0.2405],
        [ 0.0212, -0.5915, -0.5571,  0.0212, -0.5915, -0.5571,  0.0212, -0.5915,
         -0.5571]])

In [9]:
class VisionTransformer(nn.Module):
    """ Vision Transformer """

    def __init__(self, img_size=28, patch_size=4, in_chans=1, num_classes=10, embed_dim=768, depth=12,
                 num_heads=12, mlp_ratio=4., norm_layer=nn.LayerNorm, ):
        super().__init__()
        self.num_features = self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.depth = depth

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        ##############################################################################
        #                           IMPLEMENT YOUR CODE                              #
        ##############################################################################
        # similarly to cls_token, define a learnable positional embedding that matches the patchified input token size.

        #posotion embedding. 크기 : (1,총 토큰 개수, 임베딩 차원), 총 토큰 개수 : num_patches + 1
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        ##############################################################################
        #                              END YOUR CODE                                 #
        ##############################################################################

        self.blocks = nn.ModuleList([
            Block(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio,  norm_layer=norm_layer)
            for i in range(depth)])
        self.norm = norm_layer(embed_dim)

        # Classifier head
        self.head = nn.Linear(
            embed_dim, num_classes) if num_classes > 0 else nn.Identity()

    def forward(self, x):
        ##############################################################################
        #                           IMPLEMENT YOUR CODE                              #
        ##############################################################################
        B = x.shape[0] # Batch size

        # Patch Embedding
        x = self.patch_embed(x)

        # Concatenate class tokens to patch embedding
        #cls_token은 (1,1,D)라서 batch size (B)만큼 복사해야함.
        cls_tokens = self.cls_token.expand(B,-1,-1)
        x = torch.cat((cls_tokens, x), dim=1) # torch.cat으로 패치들 맨 앞에 붙임.

        # Add positional embedding to patches
        #위치 정보를 더해줌(Broadcasting으로 자동 적용)
        x = x + self.pos_embed

        # Forward through encoder blocks
        # 정의된 12개(depth)의 블록을 차례대로 통과.
        for blk in self.blocks:
            x = blk(x)

        # Use class token for classification
        # Encoder를 통과한 후, 전체 sequence 중 맨 앞에 Class Token만 뽑아옴
        # 마지막에 layer norm을 적용해야함
        x = self.norm(x)
        x = x[:, 0]

        # Classifier head
        #최종 분류
        x = self.head(x)
        ##############################################################################
        #                              END YOUR CODE                                 #
        ##############################################################################
        return x

## 2. Training a small ViT model on FashionMNIST dataset.

Define and Train a vision transformer on FashionMNIST dataset. **(You must reach above 85% for full points.)** <br>
Train with at least 5 different hyperparameter settings varying the following ViT hyperparameters.
Report the setting for the best performance.

#### ViT hyperparameters:
* patch_size
* embed_dim
* depth
* num_heads
* mlp_ratio
* etc.


In [10]:
import numpy as np

from tqdm import tqdm, trange

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader

from torchvision.transforms import ToTensor
from torchvision.datasets.mnist import FashionMNIST

In [33]:
def Train():
    ##############################################################################
    #                           IMPLEMENT YOUR CODE                              #
    ##############################################################################

    patch_size=4
    embed_dim=64
    depth=4
    num_heads= 8# make sure embed_dim is divisible by num_heads!
    mlp_ratio= 4.0

    ##############################################################################
    #                              END YOUR CODE                                 #
    ##############################################################################

    # Loading data
    transform = ToTensor()

    train_set = FashionMNIST(root='./data', train=True, download=True, transform=transform)
    test_set = FashionMNIST(root='./data', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_set, shuffle=True, batch_size=128)
    test_loader = DataLoader(test_set, shuffle=False, batch_size=128)

    # Defining model and training options
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device: ", device, f"({torch.cuda.get_device_name(device)})" if torch.cuda.is_available() else "")

    model = VisionTransformer(patch_size=patch_size, embed_dim=embed_dim, depth=depth, num_heads=num_heads, mlp_ratio=mlp_ratio).to(device)
    model_path = './vit.pth'
    N_EPOCHS = 5
    LR = 0.005

    # Training loop
    optimizer = Adam(model.parameters(), lr=LR)
    criterion = CrossEntropyLoss()
    for epoch in trange(N_EPOCHS, desc="Training"):
        train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1} in training", leave=False):
            x, y = batch
            x, y = x.to(device), y.to(device)
            y_hat = model(x)
            loss = criterion(y_hat, y)

            train_loss += loss.detach().cpu().item() / len(train_loader)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1}/{N_EPOCHS} loss: {train_loss:.2f}")

    # Test loop
    with torch.no_grad():
        correct, total = 0, 0
        test_loss = 0.0
        for batch in tqdm(test_loader, desc="Testing"):
            x, y = batch
            x, y = x.to(device), y.to(device)
            y_hat = model(x)
            loss = criterion(y_hat, y)
            test_loss += loss.detach().cpu().item() / len(test_loader)

            correct += torch.sum(torch.argmax(y_hat, dim=1) == y).detach().cpu().item()
            total += len(x)
        print(f"Test loss: {test_loss:.2f}")
        print(f"Test accuracy: {correct / total * 100:.2f}%")

    torch.save(model.state_dict(), model_path)
    print('Saved Trained Model.')

Train()

Using device:  cuda (Tesla T4)


Training:  20%|██        | 1/5 [00:12<00:49, 12.41s/it]

Epoch 1/5 loss: 0.84



Training:  40%|████      | 2/5 [00:24<00:35, 11.94s/it]

Epoch 2/5 loss: 0.49


...

Epoch 4/5 loss: 0.42



Training: 100%|██████████| 5/5 [00:59<00:00, 11.83s/it]


Epoch 5/5 loss: 0.39


Testing: 100%|██████████| 79/79 [00:01<00:00, 53.90it/s]

Test loss: 0.40
Test accuracy: 85.06%
Saved Trained Model.


### Describe what you did and discovered here
In this cell you should write all the settings tried and performances you obtained. Report what you did and what you discovered from the trials.
You can write in Korean and English

# **1. 과제 개요 및 제약사항**

이번 과제는 hyperparameter를 tuning해서 최고의 성능을 내야했습니다. 여기서 hyperparameter중에서 Learning Rate와 Epochs를 조정하는것이 성능에 가장 큰 영향을 주지만, 각각0.005와 5로 고정된 내에서 최적의 성능을 내는 것이 핵심이었습니다. Learning Rate가 높다 보니 Embed Dim을 조금만 키워도 Loss가 튀거나 급격한 성능 저하가 발생했고, Epochs가 적어서 초기부터 빠른게 성능을 높히는 가벼운 구조를 찾아야 했습니다.

결국 안정적이면서도 효율적인 모델을 확보하는 것이 목표였습니다.

# **2. 과제 과정 및 분석**

Step 1: 가장 먼저 좋은 hyperparameter 조합을 찾기 위한 기준점을 만들었습니다. Patch Size 4, Embed Dim 64, Depth 4, Num Heads 4, MLP Ratio 2.0으로 기준이 되는 모델을 학습시켰습니다. 학습 결과 Accuracy 82.05%를 기록했습니다. 이는 준수한 시작점이었으나 목표치인 85%에는 미치지 못했기 때문에, 모델의 capacity를 늘려 performance를 향상해야 한다고 판단했습니다.

Step 2: 첫 번째 튜닝 시도로 모델의 추론 능력을 강화하기 위해 Depth를 4에서 8로 두 배 늘려보았습니다. 하지만 결과는 Accuracy 80.23%로 기준 모델보다 오히려 하락했습니다. 이는 5 Epochs라는 짧은 학습 횟수 내에 깊어진 layer들이 충분히 optimize 되지 못했기 때문으로 분석했습니다. 즉, 현재 조건에서는 깊은 모델이 비효율적임을 확인했습니다.

Step 3: Depth 확장이 실패했으므로, 이번에는 Embed Dim을 64에서 128로 늘려 width를 확장하는 시도를 했습니다. 그러나 이 실험은 Accuracy 71.15%라는 가장 낮은 성능을 기록하며 실패했습니다. 원인은 Learning Rate(0.005)에 있었습니다. Embed Dim을 64에서 128로 키우자, 모델의 Parameter 개수가 갑자기 확 늘어나 모델이 너무 커지면서 Loss Landscape가 훨씬 험하고 가파르게 변한것이 원인이라고 생각했습니다.

Step 4: 모델 구조 변경의 한계를 느꼈기 때문에, 입력 단의 Patch Size를 4에서 7로 키워보았습니다. Patch Size가 커지면 이미지를 나누는 Token의 수가 줄어들어 처리속도는 매우 빨라집니다. 그 결과 Accuracy는 82.27%로 정체되었습니다. 이는 큰 Patch Size가 이미지의 데이터셋의 local detail을 뭉개버려 information loss를 유발했다고 생각했습니다.

Step 5: 앞선 실패들을 바탕으로 최종 전략을 수립했습니다. Patch Size는 다시 4로 되돌려 detail을 확보하고, Embed Dim은 64로 고정하여 학습의 안정성을 챙겼습니다. 대신 Depth나 Dim을 건드리지 않고, Num Heads를 4에서 8로, MLP Ratio를 2.0에서 4.0으로 증가시켰습니다. 이는 학습 안정성을 해치지 않는 선에서 feature extraction의 다양성과 연산량을 보강하는 방식이었습니다. 그 결과, 학습 불안정 없이 성능이 향상되어 최종 Accuracy 85.06%를 달성했습니다.

# **3. 최종 결과**

이번 과제를 통해 단순히 모델의 크기만 키우는 것이 정답은 아니라는 점을 명확히 깨달았습니다. High Learning Rate와 같은 제약 사항이 있는 환경에서 무리하게 파라미터를 늘리면, 오히려 학습이 불안정해져 loss가 오히려 커지는 현상을 볼 수 있었습니다. 따라서 향후 실무나 연구에서는 주어진 환경 내에서 학습 안정성을 먼저 확보한 뒤, Num Heads나 MLP Ratio와 같이 연산 효율이 좋은 요소들을 선별적으로 확장해 나가는 전략적인 Hyperparameter Tuning이 성능 최적화의 핵심임을 배웠습니다.